In [0]:
%sql
-- NYC WEATHER BRONZE: March-May 2026, using the team's catalog and schema.
-- Import this as a Databricks notebook; do not paste it into one SQL cell.
-- Python downloads the API data. SQL creates, loads, and checks the tables.
-- These are archived forecast values, not weather-station observations.
CREATE CATALOG IF NOT EXISTS `nyc-mobility`;
CREATE SCHEMA IF NOT EXISTS `nyc-mobility`.nyc_bronze;
SET TIME ZONE 'UTC';

In [0]:
%pip install openmeteo-requests==1.7.5 requests-cache==1.3.3 retry-requests==2.0.0
dbutils.library.restartPython()


In [0]:
import hashlib
import json
from datetime import datetime, timezone
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

# The live forecast endpoint cannot supply this past three-month window.
URL = "https://historical-forecast-api.open-meteo.com/v1/forecast"
LATITUDE, LONGITUDE = 40.7143, -74.006
START_DATE, END_DATE = "2026-03-01", "2026-05-31"
HOURLY_VARIABLES = [
    "temperature_2m", "apparent_temperature", "precipitation_probability",
    "rain", "weather_code", "cloud_cover", "visibility",
    "wind_speed_10m", "wind_gusts_10m"
]
params = {
    "latitude": LATITUDE, "longitude": LONGITUDE,
    "start_date": START_DATE, "end_date": END_DATE,
    "hourly": HOURLY_VARIABLES, "timezone": "UTC",
    "temperature_unit": "celsius", "wind_speed_unit": "kmh",
    "precipitation_unit": "mm"
}
# Memory caching avoids needing a writable .cache folder in Databricks.
cache_session = requests_cache.CachedSession(backend="memory", expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
captured = []
def remember_response(http_response, *args, **kwargs):
    captured.append(http_response)
retry_session.hooks["response"].append(remember_response)
openmeteo = openmeteo_requests.Client(session=retry_session)
responses = openmeteo.weather_api(URL, params=params, timeout=30)
if len(responses) != 1:
    raise ValueError("Expected one response for New York.")
response = responses[0]
hourly = response.Hourly()
if hourly is None or hourly.Interval() != 3600:
    raise ValueError("Missing hourly data or unexpected time interval.")
if hourly.VariablesLength() != len(HOURLY_VARIABLES):
    raise ValueError("The returned variable count differs from the request.")

# Values use the same order as the requested hourly variables.
hourly_data = {"date": pd.date_range(
    start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
    end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
    freq=pd.Timedelta(seconds=hourly.Interval()), inclusive="left"
)}
for index, name in enumerate(HOURLY_VARIABLES):
    values = hourly.Variables(index).ValuesAsNumpy()
    if len(values) != len(hourly_data["date"]):
        raise ValueError(f"Time and value lengths differ for {name}.")
    hourly_data[name] = values
hourly_dataframe = pd.DataFrame(hourly_data)
expected_dates = pd.date_range(
    START_DATE, pd.Timestamp(END_DATE) + pd.Timedelta(days=1),
    freq="h", inclusive="left", tz="UTC"
)
# Fail before loading if timestamps are missing, duplicated, or out of order.
if not pd.DatetimeIndex(hourly_dataframe["date"]).equals(expected_dates):
    raise ValueError("The API did not return the complete requested UTC window.")
if response.UtcOffsetSeconds() != 0:
    raise ValueError("Expected UTC timestamps.")
http_response = captured[-1]
http_response.raise_for_status()
# This SDK returns binary FlatBuffers. Preserve those exact response bytes.
raw_bytes = http_response.content
raw_sha256 = hashlib.sha256(raw_bytes).hexdigest()
request_json = json.dumps({"endpoint": URL, "params": params}, sort_keys=True)
batch_id = hashlib.sha256((request_json + raw_sha256).encode()).hexdigest()
ingested_at = datetime.now(timezone.utc)
print("HTTP status:", http_response.status_code)
print("Requested coordinates:", LATITUDE, LONGITUDE)
print("Returned grid coordinates:", response.Latitude(), response.Longitude())
print("Elevation (m):", response.Elevation())
print("UTC offset (seconds):", response.UtcOffsetSeconds())
print("Expected hours:", len(expected_dates), "Actual hours:", len(hourly_dataframe))
print("Duplicate timestamps:", hourly_dataframe["date"].duplicated().sum())
print("Null counts:\n", hourly_dataframe.isna().sum())
print("Response SHA-256:", raw_sha256)
display(hourly_dataframe)

In [0]:
%sql
-- One row per response: preserves the source for audit and future reprocessing.
CREATE TABLE IF NOT EXISTS `nyc-mobility`.nyc_bronze.weather_client_batches (
  batch_id STRING, request_json STRING, raw_response BINARY,
  raw_sha256 STRING, ingested_at TIMESTAMP, row_count INT
) USING DELTA;

-- One row per requested location and UTC hour: latest successfully fetched values.
-- All measurements remain numeric; validation/cleaning belongs in Silver.
CREATE TABLE IF NOT EXISTS `nyc-mobility`.nyc_bronze.weather_hourly_raw (
  weather_key STRING, date TIMESTAMP,
  temperature_2m DOUBLE, apparent_temperature DOUBLE,
  precipitation_probability DOUBLE, rain DOUBLE, weather_code DOUBLE,
  cloud_cover DOUBLE, visibility DOUBLE, wind_speed_10m DOUBLE, wind_gusts_10m DOUBLE,
  requested_latitude DOUBLE, requested_longitude DOUBLE,
  returned_latitude DOUBLE, returned_longitude DOUBLE,
  series_id STRING, batch_id STRING, ingested_at TIMESTAMP, value_hash STRING
) USING DELTA;

In [0]:
from pyspark.sql import functions as F
spark.conf.set("spark.sql.session.timeZone", "UTC")
# Explicit types also work when an entire API variable is missing/null.
# Convert missing numeric values to SQL NULL; never replace them with zero.
rows = [
    (row[0].to_pydatetime(), *[None if pd.isna(v) else float(v) for v in row[1:]])
    for row in hourly_dataframe.itertuples(index=False, name=None)
]
schema = "date TIMESTAMP, " + ", ".join(name + " DOUBLE" for name in HOURLY_VARIABLES)
weather_df = spark.createDataFrame(rows, schema=schema)
weather_df = (weather_df
    .withColumn("requested_latitude", F.lit(LATITUDE))
    .withColumn("requested_longitude", F.lit(LONGITUDE))
    .withColumn("returned_latitude", F.lit(float(response.Latitude())))
    .withColumn("returned_longitude", F.lit(float(response.Longitude())))
    .withColumn("series_id", F.lit("historical_forecast:best_match"))
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn("ingested_at", F.lit(ingested_at))
    .withColumn("weather_key", F.sha2(F.concat_ws("|",
        F.lit(f"{LATITUDE},{LONGITUDE}"), F.col("series_id"),
        F.col("date").cast("long")), 256))
    .withColumn("value_hash", F.sha2(F.to_json(F.struct(
        *HOURLY_VARIABLES, "returned_latitude", "returned_longitude")), 256)))
weather_df.createOrReplaceTempView("weather_incoming")
batch_df = spark.createDataFrame(
    [(batch_id, request_json, raw_bytes, raw_sha256, ingested_at, len(rows))],
    "batch_id STRING, request_json STRING, raw_response BINARY, raw_sha256 STRING, ingested_at TIMESTAMP, row_count INT")
batch_df.createOrReplaceTempView("weather_batch_incoming")
# Stop safely if an earlier draft created a different table schema.
for table, incoming in [("weather_hourly_raw", weather_df), ("weather_client_batches", batch_df)]:
    existing = spark.table(f"`nyc-mobility`.nyc_bronze.{table}")
    if dict(existing.dtypes) != dict(incoming.dtypes):
        raise ValueError(f"Review existing {table} schema before loading; no tables were dropped.")

In [0]:
%sql
-- Rerunning does not insert another copy of an identical response.
MERGE INTO `nyc-mobility`.nyc_bronze.weather_client_batches AS target
USING weather_batch_incoming AS source ON target.batch_id = source.batch_id
WHEN NOT MATCHED THEN INSERT *;

-- Keep one current row per location/series/hour, even when rerunning the notebook.
-- Original response versions remain in weather_client_batches. Use one writer.
MERGE INTO `nyc-mobility`.nyc_bronze.weather_hourly_raw AS target
USING weather_incoming AS source ON target.weather_key = source.weather_key
WHEN MATCHED AND target.value_hash <> source.value_hash
  AND source.ingested_at >= target.ingested_at THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
-- Reconcile this run against the saved table. Both counts must be 2,208.
SELECT COUNT(*) AS incoming_hours,
       COUNT(saved.weather_key) AS matched_bronze_hours,
       COUNT_IF(saved.value_hash = incoming.value_hash) AS matching_values
FROM weather_incoming incoming
LEFT JOIN `nyc-mobility`.nyc_bronze.weather_hourly_raw saved
  ON saved.weather_key = incoming.weather_key;

In [0]:
%sql
-- This query should return no rows.
SELECT weather_key, COUNT(*) AS copies
FROM `nyc-mobility`.nyc_bronze.weather_hourly_raw
GROUP BY weather_key HAVING COUNT(*) > 1;

In [0]:
%sql
-- Expected monthly totals in UTC: March 744, April 720, May 744.
SELECT date_format(date, 'yyyy-MM') AS month, COUNT(*) AS hourly_rows
FROM weather_incoming GROUP BY date_format(date, 'yyyy-MM') ORDER BY month;

In [0]:
%sql
SELECT date, temperature_2m, apparent_temperature, precipitation_probability,
       rain, weather_code, cloud_cover, visibility, wind_speed_10m, wind_gusts_10m
FROM `nyc-mobility`.nyc_bronze.weather_hourly_raw
WHERE requested_latitude = 40.7143 AND requested_longitude = -74.006
  AND series_id = 'historical_forecast:best_match'
  AND date >= TIMESTAMP '2026-03-01 00:00:00'
  AND date < TIMESTAMP '2026-06-01 00:00:00'
ORDER BY date LIMIT 5;